# Nutrition Coach on Kaggle

1. Phone-verify your Kaggle account (Settings → Phone verification). Kaggle needs this for GPUs and internet.
2. In this notebook's right sidebar, **Session options**: Accelerator → **GPU T4 x2**, Internet → **On**.
3. **Add-ons → Secrets**: add `NGROK_AUTH_TOKEN` and, optionally, `FDC_API_KEY`, and tick them for this notebook.
4. Check `REPO_URL` below, then run both cells.
5. When you're done, click **Stop session** (power icon) so you don't use up your weekly GPU hours.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/woyeta/NutritionCoach.git"
MODEL_ID = "google/gemma-4-E4B-it"  # or "google/gemma-4-E2B-it" (smaller, faster)

!rm -rf /kaggle/working/NutritionCoach && git clone -q $REPO_URL /kaggle/working/NutritionCoach
%cd /kaggle/working/NutritionCoach

# Kaggle already has torch. Gemma 4 needs a recent transformers, so upgrade it.
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q flask pyngrok "pydantic>=2.6" requests sentencepiece protobuf

secrets = UserSecretsClient()
os.environ["MODEL_ID"] = MODEL_ID
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use one of the two T4s; the model fits on one
os.environ["NGROK_AUTH_TOKEN"] = secrets.get_secret("NGROK_AUTH_TOKEN")
try:
    os.environ["FDC_API_KEY"] = secrets.get_secret("FDC_API_KEY")
except Exception:
    print("No FDC_API_KEY secret: using the rate-limited USDA DEMO_KEY")
print("Setup complete")

In [ ]:
import socket
import subprocess
import time
from pyngrok import conf, ngrok

# Safe to re-run: stop the previous app and tunnel first.
if "server" in globals():
    server.kill()
ngrok.kill()

conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]

server = subprocess.Popen(["python", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait until the app answers on 127.0.0.1:5000 (or stop if it crashed).
for _ in range(60):
    if server.poll() is not None:
        print(server.stdout.read().decode())
        raise RuntimeError("The app crashed on startup; see the error above.")
    try:
        socket.create_connection(("127.0.0.1", 5000), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("The app did not start within 60 seconds.")

# 127.0.0.1, not "localhost": on Kaggle localhost can resolve to IPv6 (::1),
# where the app is not listening.
public_url = ngrok.connect("http://127.0.0.1:5000")  # full URL: pyngrok prefixes "localhost:" otherwise
print(f"Nutrition Coach is live at: {public_url}")
print("The first question takes a few minutes while the model downloads and loads.\n")

for line in server.stdout:
    print(line.decode(), end="")
